In [1]:
# ===== 기본 세트 (거의 항상 필요) =====
import pandas as pd                    # 데이터프레임 다루기 (표 형태 데이터 처리)
import numpy as np                     # 수치 계산, 배열 연산
import matplotlib.pyplot as plt        # 기본 시각화
import seaborn as sns                  # matplotlib 기반의, 더 예쁘고 간결한 통계 시각화

# ===== 파일/경로/시스템 =====
import os                              # 파일 경로 확인, 파일 존재 여부(os.path.exists) 등
from pathlib import Path               # 폴더/파일 경로를 객체로 다루기 (work_dir / "파일명")
import joblib                          # 학습된 scaler/encoder를 파일로 저장하고 재사용
import platform                        # OS 종류 확인 (Windows/Mac/Linux 구분 필요할 때)
import warnings                        # 경고 메시지 제어(숨기기/표시)

# ===== 벤치마크/성능 측정 =====
import time                            # 코드 실행 시간 측정 (time.perf_counter())
import psutil                          # 메모리 사용량 측정 (rss_mb 등)
import gc                              # 메모리 강제 정리(garbage collection)

# ===== 결측치 탐색 시각화 =====
import missingno as msno               # 결측치 위치(matrix), 관계(heatmap) 시각화

# ===== 인터랙티브 시각화 =====
import plotly.express as px            # 마우스 hover/zoom 가능한 인터랙티브 그래프
import plotly.io as pio                 # plotly 렌더러(출력 방식) 설정

# ===== 대용량 데이터 처리 =====
import polars as pl                     # pandas보다 빠른 대용량 데이터 처리 (Rust 기반)

# ===== 통계 검증 =====
import statsmodels.api as sm            # 회귀분석, 다중공선성(Cond. No.) 등 통계 검증
from scipy import stats
from statsmodels.stats.power import TTestIndPower
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors

# ===== 한글 폰트 설정 =====
import matplotlib.font_manager as fm    # 그래프에 한글 폰트 적용 시 필요
import matplotlib.pyplot as plt

font_path = "C:/Windows/Fonts/malgun.ttf"
fm.fontManager.addfont(font_path)
plt.rcParams["font.family"] = fm.FontProperties(fname=font_path).get_name()
plt.rcParams["axes.unicode_minus"] = False

# ===== 인코딩/스케일링/모델링 (sklearn) =====
from sklearn.preprocessing import LabelEncoder      # 카테고리를 숫자로 변환 (순서형)
from sklearn.preprocessing import MinMaxScaler       # 0~1 범위로 스케일링
from sklearn.preprocessing import StandardScaler     # 평균0, 표준편차1로 스케일링
from sklearn.preprocessing import RobustScaler       # 이상치에 강한 스케일링 (median/IQR 기반)
from sklearn.linear_model import LinearRegression    # 선형 회귀 모델
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀(분류) 모델
from sklearn.ensemble import RandomForestRegressor   # 랜덤포레스트(트리 기반) 모델
from sklearn.metrics import roc_auc_score, log_loss  # 분류 모델 평가지표

In [2]:
cookie_cats = pd.read_csv("cookie_cats.csv")

In [3]:
cookie_cats.info()

# userid 플레이어 아이디(고유)
# version 게이트 구분(30/40)
# sum_gamerounds 설치 후 14일 동안 플레이어가 플레이한 게임 라운드 수
# retention_1 설치 후 하루만에 재접속 여부
# retention_7 설치 후 7일만에 재접속 여부

<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   userid          90189 non-null  int64
 1   version         90189 non-null  str  
 2   sum_gamerounds  90189 non-null  int64
 3   retention_1     90189 non-null  bool 
 4   retention_7     90189 non-null  bool 
dtypes: bool(2), int64(2), str(1)
memory usage: 2.8 MB


In [4]:
cookie_cats['version'].value_counts()

version
gate_40    45489
gate_30    44700
Name: count, dtype: int64

In [5]:
cookie_cats['sum_gamerounds'].value_counts()

sum_gamerounds
1       5538
2       4606
0       3994
3       3958
4       3629
        ... 
572        1
2063       1
846        1
768        1
708        1
Name: count, Length: 942, dtype: int64

In [6]:
cookie_cats['retention_1'].value_counts()

retention_1
False    50036
True     40153
Name: count, dtype: int64

In [7]:
cookie_cats['retention_7'].value_counts()

retention_7
False    73408
True     16781
Name: count, dtype: int64

In [8]:
# userid 중복 확인
cookie_cats[cookie_cats.duplicated('userid')]


,userid,version,sum_gamerounds,retention_1,retention_7


In [9]:
# sum_gamerounds 결측 확인
cookie_cats[cookie_cats['sum_gamerounds'].isna()]

,userid,version,sum_gamerounds,retention_1,retention_7


In [10]:
# sum_gamerounds 이상치 확인
Q1 = cookie_cats["sum_gamerounds"].quantile(0.25)
Q3 = cookie_cats["sum_gamerounds"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = cookie_cats[(cookie_cats["sum_gamerounds"] < lower ) | (cookie_cats["sum_gamerounds"] > upper)]

print(f"Q1: {Q1}, Q3: {Q3}, IQR = {IQR}")
print(f"정상 범위: {lower} ~ {upper}")
print(f"이상치 개수: {len(outliers)}개")
print(outliers)


Q1: 5.0, Q3: 51.0, IQR = 46.0
정상 범위: -64.0 ~ 120.0
이상치 개수: 10177개
        userid  version  sum_gamerounds  retention_1  retention_7
2          377  gate_40             165         True        False
4          488  gate_40             179         True         True
5          540  gate_40             187         True         True
9         1587  gate_40             153         True        False
14        2218  gate_30             305         True         True
...        ...      ...             ...          ...          ...
90121  9991145  gate_30             328         True         True
90125  9991408  gate_40             186         True         True
90134  9991949  gate_30             191         True         True
90150  9995412  gate_40             253         True         True
90160  9996269  gate_30             143        False        False

[10177 rows x 5 columns]


In [11]:
# 정규성 검정
gate_30 = cookie_cats.loc[cookie_cats['version'] == 'gate_30', 'sum_gamerounds']
gate_40 = cookie_cats.loc[cookie_cats['version'] == 'gate_40', 'sum_gamerounds']

gate_30, p_30 = stats.shapiro(gate_30.sample(min(5000, len(gate_30)), random_state = 42))
gate_40, p_40 = stats.shapiro(gate_40.sample(min(5000, len(gate_40)), random_state = 42))

print(f"gate_30 정규성 검정 p-값: {p_30:.10f}")
print(f"gate_40 정규성 검정 p-값: {p_40:.10f}")

gate_30 정규성 검정 p-값: 0.0000000000
gate_40 정규성 검정 p-값: 0.0000000000


In [12]:
# 등분산성 검정
gate_30 = cookie_cats.loc[cookie_cats['version'] == 'gate_30', 'sum_gamerounds']
gate_40 = cookie_cats.loc[cookie_cats['version'] == 'gate_40', 'sum_gamerounds']

levene_stat, levene_p = stats.levene(gate_30, gate_40)
print(f"\nLevene 등분산성 p-값: {levene_p:.10f}")


Levene 등분산성 p-값: 0.4669451677


In [13]:
from scipy import stats

gate_30 = cookie_cats.loc[cookie_cats["version"] == "gate_30", "sum_gamerounds"]
gate_40 = cookie_cats.loc[cookie_cats["version"] == "gate_40", "sum_gamerounds"]

# 대립가설: "gate_40이 gate_30보다 크다" -> gate_40을 먼저, alternative='greater'
u_stat, p_value = stats.mannwhitneyu(gate_40, gate_30, alternative="greater")

print(f"U-통계량: {u_stat}")
print(f"p-값: {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print(f"→ p-value < {alpha}: 귀무가설 기각. gate_40이 gate_30보다 유의하게 더 오래 플레이한다.")
else:
    print(f"→ p-value >= {alpha}: 귀무가설을 기각하지 못한다.")

U-통계량: 1009027049.5
p-값: 0.974896
→ p-value >= 0.05: 귀무가설을 기각하지 못한다.


In [14]:
import numpy as np

np.random.seed(42)

# 1. 점추정: 중앙값 차이 (Mann-Whitney와 짝을 이루는 지표)
median_diff = gate_40.median() - gate_30.median()

# 2. 95% 신뢰구간: 부트스트랩으로 계산
n_boot = 2000
boot_diffs = []
for i in range(n_boot):
    sample_30 = gate_30.sample(len(gate_30), replace=True)
    sample_40 = gate_40.sample(len(gate_40), replace=True)
    boot_diffs.append(sample_40.median() - sample_30.median())

ci_lower = np.percentile(boot_diffs, 2.5)
ci_upper = np.percentile(boot_diffs, 97.5)

# 3. 효과크기: rank-biserial correlation
n1, n2 = len(gate_40), len(gate_30)
r_effect = 1 - (2 * u_stat) / (n1 * n2)

print(f"1. 점추정(중앙값 차이, gate_40-gate_30): {median_diff:.2f}")
print(f"2. 95% 신뢰구간: ({ci_lower:.2f}, {ci_upper:.2f})")
print(f"3. 효과크기(rank-biserial r): {r_effect:.4f}")

1. 점추정(중앙값 차이, gate_40-gate_30): -1.00
2. 95% 신뢰구간: (-1.00, 0.00)
3. 효과크기(rank-biserial r): 0.0075
